# SMS Spam Detection Using Machine Learning

This project implements an SMS spam detection system using machine learning.

### Methodology

1. **Pre-processing**
   - Text cleaning
   - Tokenisation
   - Stopword removal
   - Stemming

2. **Feature Representation**
   - Bag of Words (BoW)
   - TF-IDF
   - Unigrams and bigrams

3. **Model Training**
   - Naive Bayes
   - Support Vector Machine (SVM)
   - Logistic Regression
   - Random Forest
   - Grid Search with 10-fold cross-validation

4. **Evaluation**
   - Accuracy, precision, recall and F1-score
   - Confusion matrices
   - Important features
   - Wilcoxon signed-rank test

### Dataset

The **UCI SMS Spam Collection** dataset contains 5,574 SMS messages:

- 4,827 ham messages
- 747 spam messages

Dataset: [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection)

The dataset file is `spam.csv`, with `v1` as the label and `v2` as the message text.

In [1]:
# Libraries needed
import argparse
import json
import re
import string
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from scipy.stats import wilcoxon

try:
    import nltk
    from nltk.corpus import stopwords as nltk_stopwords
    from nltk.stem import PorterStemmer
    from nltk.tokenize import word_tokenize as nltk_word_tokenize
    NLTK_AVAILABLE = True
except ImportError:
    NLTK_AVAILABLE = False

warnings.filterwarnings("ignore")

# Minimal fallback stopword list used only if NLTK / its corpora are unavailable. If no internet access to run nltk.download. Prefer NLTK when possible.
FALLBACK_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by", "can",
    "did", "do", "does", "doing", "for", "from", "had", "has", "have",
    "having", "he", "her", "here", "hers", "herself", "him", "himself",
    "his", "how", "i", "if", "in", "into", "is", "it", "its", "itself",
    "just", "me", "more", "most", "my", "myself", "no", "nor", "not",
    "of", "on", "or", "our", "ours", "ourselves", "out", "over", "own",
    "s", "same", "she", "should", "so", "some", "such", "t", "than",
    "that", "the", "their", "theirs", "them", "themselves", "then",
    "there", "these", "they", "this", "those", "through", "to", "too",
    "under", "until", "up", "very", "was", "we", "were", "what", "when",
    "where", "which", "while", "who", "whom", "why", "will", "with",
    "you", "your", "yours", "yourself", "yourselves", "am", "or",
}

RANDOM_SEED = 42
N_FOLDS = 10
TEST_SIZE = 0.20

# Spam-indicative tokens retained despite being generic stop words (Delany et al., 2012)
KEEP_TOKENS = {"free", "win", "won", "urgent", "prize", "cash", "claim", "call", "txt"}

In [3]:
# --------------------------------------------------------------------------- #
# Stage 1: Pre-processing
# --------------------------------------------------------------------------- #
def ensure_nltk_resources():
    """Attempt to download required NLTK corpora. Safe to fail silently —
    build_preprocessor() falls back to a lightweight built-in tokenizer/
    stopword list if NLTK or its data is unavailable (e.g. no internet)."""
    if not NLTK_AVAILABLE:
        return False
    ok = True
    for pkg in ["punkt", "punkt_tab", "stopwords"]:
        try:
            nltk.data.find(f"tokenizers/{pkg}" if "punkt" in pkg else f"corpora/{pkg}")
        except LookupError:
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                ok = False
    return ok


def _simple_stem(token: str) -> str:
    """Lightweight Porter-style suffix stripping used only as a fallback
    when NLTK's PorterStemmer is unavailable."""
    for suffix in ("ing", "edly", "ed", "ies", "es", "s", "ly"):
        if token.endswith(suffix) and len(token) - len(suffix) > 2:
            return token[: -len(suffix)]
    return token


def build_preprocessor():
    nltk_ready = ensure_nltk_resources()
    use_nltk = NLTK_AVAILABLE and nltk_ready

    if use_nltk:
        try:
            stop_words = set(nltk_stopwords.words("english")) - KEEP_TOKENS
            stemmer = PorterStemmer()
            tokenize = nltk_word_tokenize
            stem = stemmer.stem
        except Exception:
            use_nltk = False

    if not use_nltk:
        stop_words = FALLBACK_STOPWORDS - KEEP_TOKENS
        tokenize = str.split
        stem = _simple_stem

    url_re = re.compile(r"http\S+|www\.\S+")
    num_re = re.compile(r"\d+")
    punct_table = str.maketrans(
        "", "", string.punctuation.replace("£", "").replace("$", "")
    )

    def clean(text: str) -> str:
        text = text.lower()
        text = url_re.sub(" ", text)
        text = text.translate(punct_table)
        text = num_re.sub(" ", text)
        tokens = tokenize(text)
        tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
        tokens = [stem(t) for t in tokens]
        return " ".join(tokens)

    return clean

In [4]:
# --------------------------------------------------------------------------- #
# Stage 2: Feature representations
# --------------------------------------------------------------------------- #
def get_representations():
    return {
        "BoW_unigram":     CountVectorizer(ngram_range=(1, 1), min_df=2),
        "BoW_uni+bi":      CountVectorizer(ngram_range=(1, 2), min_df=2),
        "TFIDF_unigram":   TfidfVectorizer(ngram_range=(1, 1), min_df=2),
        "TFIDF_uni+bi":    TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    }


In [5]:
# --------------------------------------------------------------------------- #
# Stage 3: Classifiers and hyperparameter grids
# --------------------------------------------------------------------------- #
def get_classifiers():
    return {
        "NaiveBayes": (
            MultinomialNB(),
            {"alpha": [0.1, 0.5, 1.0]}
        ),
        "SVM": (
            SVC(kernel="linear", probability=True, random_state=RANDOM_SEED),
            {"C": [0.1, 1, 10]}
        ),
        "LogisticRegression": (
            LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
            {"C": [0.1, 1, 10], "penalty": ["l2"]}
        ),
        "RandomForest": (
            RandomForestClassifier(random_state=RANDOM_SEED),
            {"n_estimators": [200, 400], "max_depth": [None, 30]}
        ),
    }

In [6]:
# --------------------------------------------------------------------------- #
# Stage 4: Evaluation
# --------------------------------------------------------------------------- #
def false_positive_rate(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return fp / (fp + tn) if (fp + tn) > 0 else 0.0


def evaluate(y_true, y_pred, y_score):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score) if y_score is not None else np.nan,
        "fpr": false_positive_rate(y_true, y_pred),
    }


def top_features(vectorizer, clf, clf_name, n=20):
    """Return the n words/bigrams most associated with the spam class."""
    feature_names = np.array(vectorizer.get_feature_names_out())
    if clf_name == "NaiveBayes":
        scores = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
    elif clf_name in ("SVM", "LogisticRegression"):
        coef = clf.coef_
        scores = coef.toarray().ravel() if hasattr(coef, "toarray") else np.ravel(coef)
    elif clf_name == "RandomForest":
        scores = clf.feature_importances_
    else:
        return []
    top_idx = np.argsort(scores)[-n:][::-1]
    return list(zip(feature_names[top_idx], scores[top_idx]))

In [7]:
# --------------------------------------------------------------------------- #
# Main experiment loop
# --------------------------------------------------------------------------- #
def run_experiments(df: pd.DataFrame, outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)

    print("Stage 1: pre-processing...")
    clean = build_preprocessor()
    df["clean_text"] = df["text"].apply(clean)

    X_train_text, X_test_text, y_train, y_test = train_test_split(
        df["clean_text"], df["label"], test_size=TEST_SIZE,
        stratify=df["label"], random_state=RANDOM_SEED
    )

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    representations = get_representations()
    classifiers = get_classifiers()

    results = []
    cv_f1_scores = {}          # for Wilcoxon test: config -> list of per-fold F1
    fitted_artifacts = {}      # config -> (vectorizer, best_estimator) for explainability

    print("Stage 2-3: feature extraction + model training/tuning (16 configurations)...")
    for rep_name, vectorizer_template in representations.items():
        for clf_name, (base_clf, param_grid) in classifiers.items():
            config_name = f"{clf_name}__{rep_name}"
            print(f"  -> {config_name}")

            vectorizer = vectorizer_template.__class__(**vectorizer_template.get_params())
            X_train_vec = vectorizer.fit_transform(X_train_text)
            X_test_vec = vectorizer.transform(X_test_text)

            grid = GridSearchCV(
                base_clf, param_grid, scoring="f1", cv=skf, n_jobs=-1
            )
            grid.fit(X_train_vec, y_train)
            best_clf = grid.best_estimator_

            # Collect per-fold F1 for the best hyperparameters (for significance testing)
            fold_f1 = []
            for train_idx, val_idx in skf.split(X_train_vec, y_train):
                clf_fold = best_clf.__class__(**best_clf.get_params())
                clf_fold.fit(X_train_vec[train_idx], y_train.iloc[train_idx])
                pred_fold = clf_fold.predict(X_train_vec[val_idx])
                fold_f1.append(f1_score(y_train.iloc[val_idx], pred_fold, zero_division=0))
            cv_f1_scores[config_name] = fold_f1

            # Final held-out test evaluation
            y_pred = best_clf.predict(X_test_vec)
            if hasattr(best_clf, "predict_proba"):
                y_score = best_clf.predict_proba(X_test_vec)[:, 1]
            else:
                y_score = None
            metrics = evaluate(y_test, y_pred, y_score)
            metrics.update({
                "classifier": clf_name,
                "representation": rep_name,
                "config": config_name,
                "best_params": grid.best_params_,
                "cv_f1_mean": float(np.mean(fold_f1)),
                "cv_f1_std": float(np.std(fold_f1)),
            })
            results.append(metrics)

            fitted_artifacts[config_name] = (vectorizer, best_clf, y_test, y_pred)

    results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
    results_df.to_csv(outdir / "configuration_results.csv", index=False)
    print("\nTop 5 configurations by test F1-score:")
    print(results_df[["config", "accuracy", "precision", "recall", "f1", "roc_auc", "fpr"]].head(5).to_string(index=False))

    # ---------------- Stage 4a: statistical significance (Wilcoxon) ---------------- #
    best_config = results_df.iloc[0]["config"]
    print(f"\nStage 4: statistical comparison of top configurations against best ({best_config})...")
    sig_results = {}
    for config_name, fold_scores in cv_f1_scores.items():
        if config_name == best_config:
            continue
        try:
            stat, p_value = wilcoxon(cv_f1_scores[best_config], fold_scores)
            sig_results[config_name] = {"statistic": float(stat), "p_value": float(p_value)}
        except ValueError:
            sig_results[config_name] = {"statistic": None, "p_value": None}
    with open(outdir / "significance_tests.json", "w") as f:
        json.dump({"reference_config": best_config, "comparisons": sig_results}, f, indent=2)

    # ---------------- Stage 4b: confusion matrix + top features for best config ---------------- #
    vec_best, clf_best, y_true_best, y_pred_best = fitted_artifacts[best_config]
    cm = confusion_matrix(y_true_best, y_pred_best, labels=[0, 1])
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix – {best_config}")
    plt.tight_layout()
    plt.savefig(outdir / "confusion_matrix_best.png", dpi=200)
    plt.close()

    clf_short = best_config.split("__")[0]
    features = top_features(vec_best, clf_best, clf_short, n=20)
    pd.DataFrame(features, columns=["feature", "score"]).to_csv(
        outdir / "top_features_best_config.csv", index=False
    )

    # ---------------- Summary bar chart across all configurations ---------------- #
    plt.figure(figsize=(10, 5))
    order = results_df.sort_values("f1", ascending=False)
    sns.barplot(data=order, x="config", y="f1", hue="representation", dodge=False)
    plt.xticks(rotation=75, ha="right")
    plt.ylabel("Test F1-score")
    plt.title("F1-score by Classifier–Representation Configuration")
    plt.tight_layout()
    plt.savefig(outdir / "f1_comparison_all_configs.png", dpi=200)
    plt.close()

    print(f"\nAll outputs written to: {outdir.resolve()}")
    return results_df


def load_dataset(path: str) -> pd.DataFrame:
    """Loads the UCI SMS Spam Collection CSV (v1=label, v2=text) or a generic
    2-column {label,text} CSV, and maps labels to {ham:0, spam:1}."""
    raw = pd.read_csv(path, encoding="latin-1")
    if {"v1", "v2"}.issubset(raw.columns):
        df = raw[["v1", "v2"]].rename(columns={"v1": "label", "v2": "text"})
    elif {"label", "text"}.issubset(raw.columns):
        df = raw[["label", "text"]]
    else:
        # fall back: assume first two columns are label, text
        df = raw.iloc[:, :2]
        df.columns = ["label", "text"]
    df["label"] = df["label"].str.strip().str.lower().map({"ham": 0, "spam": 1})
    df = df.dropna(subset=["label", "text"]).reset_index(drop=True)
    df["label"] = df["label"].astype(int)
    return df

In [10]:
np.random.seed(RANDOM_SEED)

DATA_PATH = "spam.csv"
OUTPUT_DIR = "results"

dataset = load_dataset(DATA_PATH)

print(f"Loaded {len(dataset)} messages "
      f"({(dataset['label'] == 1).sum()} spam, "
      f"{(dataset['label'] == 0).sum()} ham)")

run_experiments(dataset, Path(OUTPUT_DIR))

Loaded 5572 messages (747 spam, 4825 ham)
Stage 1: pre-processing...
Stage 2-3: feature extraction + model training/tuning (16 configurations)...
  -> NaiveBayes__BoW_unigram
  -> SVM__BoW_unigram
  -> LogisticRegression__BoW_unigram
  -> RandomForest__BoW_unigram
  -> NaiveBayes__BoW_uni+bi
  -> SVM__BoW_uni+bi
  -> LogisticRegression__BoW_uni+bi
  -> RandomForest__BoW_uni+bi
  -> NaiveBayes__TFIDF_unigram
  -> SVM__TFIDF_unigram
  -> LogisticRegression__TFIDF_unigram
  -> RandomForest__TFIDF_unigram
  -> NaiveBayes__TFIDF_uni+bi
  -> SVM__TFIDF_uni+bi
  -> LogisticRegression__TFIDF_uni+bi
  -> RandomForest__TFIDF_uni+bi

Top 5 configurations by test F1-score:
                          config  accuracy  precision   recall       f1  roc_auc      fpr
              SVM__TFIDF_unigram  0.986547   1.000000 0.899329 0.946996 0.981540 0.000000
  LogisticRegression__BoW_uni+bi  0.985650   1.000000 0.892617 0.943262 0.981835 0.000000
               SVM__TFIDF_uni+bi  0.984753   0.978261 0.9060

,accuracy,precision,recall,f1,roc_auc,fpr,classifier,representation,config,best_params,cv_f1_mean,cv_f1_std
9,0.986547,1.000000,0.899329,0.946996,0.981540,0.000000,SVM,TFIDF_unigram,SVM__TFIDF_unigram,{'C': 1},0.912524,0.018654
6,0.985650,1.000000,0.892617,0.943262,0.981835,0.000000,LogisticRegression,BoW_uni+bi,LogisticRegression__BoW_uni+bi,"{'C': 10, 'penalty': 'l2'}",0.925892,0.019813
13,0.984753,0.978261,0.906040,0.940767,0.982985,0.003106,SVM,TFIDF_uni+bi,SVM__TFIDF_uni+bi,{'C': 1},0.929427,0.017474
2,0.984753,0.992537,0.892617,0.939929,0.980369,0.001035,LogisticRegression,BoW_unigram,LogisticRegression__BoW_unigram,"{'C': 10, 'penalty': 'l2'}",0.928984,0.023383
14,0.982960,0.977941,0.892617,0.933333,0.984573,0.003106,LogisticRegression,TFIDF_uni+bi,LogisticRegression__TFIDF_uni+bi,"{'C': 10, 'penalty': 'l2'}",0.921269,0.020619
10,0.981166,0.977612,0.879195,0.925795,0.981703,0.003106,LogisticRegression,TFIDF_unigram,LogisticRegression__TFIDF_unigram,"{'C': 10, 'penalty': 'l2'}",0.916233,0.023995
5,0.981166,1.000000,0.859060,0.924188,0.978292,0.000000,SVM,BoW_uni+bi,SVM__BoW_uni+bi,{'C': 0.1},0.920295,0.028050
8,0.980269,0.984733,0.865772,0.921429,0.981599,0.002070,NaiveBayes,TFIDF_unigram,NaiveBayes__TFIDF_unigram,{'alpha': 0.1},0.927737,0.020315
0,0.979372,0.956522,0.885906,0.919861,0.974603,0.006211,NaiveBayes,BoW_unigram,NaiveBayes__BoW_unigram,{'alpha': 0.1},0.932235,0.020746
4,0.979372,0.963235,0.879195,0.919298,0.972804,0.005176,NaiveBayes,BoW_uni+bi,NaiveBayes__BoW_uni+bi,{'alpha': 1.0},0.937470,0.012191
